### **Hash Functions: MD5, SHA-256, SHA-512, and the Birthday Attack**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jrafa1607/Cryptography_In_Python/blob/main/Algorithms/Hash_Functions.ipynb)

> A **cryptographic hash function** maps any input to a fixed-size digest. It must be **one-way**, **collision-resistant**, and show the **avalanche effect**. **MD5** (1991) and **SHA-1** are broken for collisions; the **SHA-2** family (SHA-256, SHA-512) is the current standard. This notebook compares the functions, measures the avalanche effect, and finds a real collision on a truncated hash with the **birthday attack**.

In [1]:
# Digest sizes and the avalanche effect (one character changed)
import hashlib
import hmac
import secrets
import time

def bits_changed(a, b):
    return sum(bin(x ^ y).count("1") for x, y in zip(a, b))

m1, m2 = b"Hello world!", b"Hello world"
print(f"{'function':<8} | {'bits':>4} | {'hex chars':>9} | {'bits changed':>12}")
for name in ("md5", "sha1", "sha256", "sha512"):
    d1, d2 = hashlib.new(name, m1).digest(), hashlib.new(name, m2).digest()
    print(f"{name:<8} | {len(d1) * 8:>4} | {len(d1) * 2:>9} | {bits_changed(d1, d2):>5} of {len(d1) * 8:<4}")
print("\nSHA-256('Hello world!') =", hashlib.sha256(m1).hexdigest())

function | bits | hex chars | bits changed
md5      |  128 |        32 |    56 of 128 
sha1     |  160 |        40 |    73 of 160 
sha256   |  256 |        64 |   135 of 256 
sha512   |  512 |       128 |   243 of 512 

SHA-256('Hello world!') = c0535e4be2b79ffd93291305436bf889314e4a3faec05ecffcbb7df31ad9e51a


#### **01. The Birthday Attack**

For an `n`-bit hash, finding **any** two inputs with the same digest takes only about `2^(n/2)` attempts (the birthday paradox), not `2^n`. The demo truncates SHA-256 to 32 bits, so a collision appears after about 2^16 = 65,536 hashes. The same arithmetic says that a generic collision search on MD5 needs 2^64 attempts, but cryptanalysis does it in **seconds**: MD5 collisions forged a rogue certificate authority in 2008 and signed the Flame malware in 2012; SHA-1 fell in practice in 2017 (SHAttered).

In [2]:
# Find a collision on SHA-256 truncated to 32 bits
TRUNCATED_BITS = 32

def short_hash(data):
    return hashlib.sha256(data).digest()[:TRUNCATED_BITS // 8]

seen = {}
start = time.perf_counter()
for attempt in range(1, 2 ** 24):
    candidate = f"invoice-{attempt}".encode()
    digest = short_hash(candidate)
    if digest in seen:
        break
    seen[digest] = candidate
elapsed = time.perf_counter() - start

print(f"Collision after {attempt:,} hashes ({elapsed:.2f} s); expected about {2 ** (TRUNCATED_BITS // 2):,}")
print(f"{seen[digest].decode():>16} -> {digest.hex()}")
print(f"{candidate.decode():>16} -> {short_hash(candidate).hex()}")
assert seen[digest] != candidate and short_hash(seen[digest]) == short_hash(candidate)

Collision after 86,732 hashes (0.07 s); expected about 65,536
   invoice-70535 -> 38c845de
   invoice-86732 -> 38c845de


#### **02. Hashes Are Not Enough for Passwords or Integrity**

* **Passwords:** a plain SHA-256 is computed billions of times per second on GPUs. Use a slow, salted, memory-hard function such as **scrypt** or **Argon2**.
* **Integrity with a secret:** `SHA-256(key + message)` is vulnerable to length extension. Use **HMAC**, and compare tags in constant time.

In [3]:
# scrypt for passwords, HMAC for message authentication
password, salt = b"correct horse battery staple", secrets.token_bytes(16)
start = time.perf_counter()
stored = hashlib.scrypt(password, salt=salt, n=2 ** 14, r=8, p=1)
print(f"scrypt: {stored.hex()[:32]}... ({(time.perf_counter() - start) * 1000:.0f} ms per guess)")

start = time.perf_counter()
for _ in range(100_000):
    hashlib.sha256(password).digest()
print(f"plain SHA-256: {(time.perf_counter() - start) / 100_000 * 1e6:.2f} µs per guess (on one CPU core)")

key = secrets.token_bytes(32)
tag = hmac.new(key, b"amount=100", hashlib.sha256).digest()
print("HMAC valid    :", hmac.compare_digest(tag, hmac.new(key, b"amount=100", hashlib.sha256).digest()))
print("HMAC tampered :", hmac.compare_digest(tag, hmac.new(key, b"amount=900", hashlib.sha256).digest()))

scrypt: ea5961c78e6bc723778dd0d2d51a4016... (43 ms per guess)
plain SHA-256: 0.32 µs per guess (on one CPU core)
HMAC valid    : True
HMAC tampered : False


#### **Key Takeaways**

* Use SHA-256 or SHA-512 (or SHA-3); never MD5 or SHA-1 where collisions matter (signatures, certificates).
* An `n`-bit hash offers only `n/2` bits of collision resistance.
* Store passwords with scrypt or Argon2, and authenticate messages with HMAC.